In [ ]:
using Pkg
# adjust toktox_path to match the location of your cloned TokToxStability repo
toktox_path = "/Users/adrianaghiozzi/.julia/dev/TokToxStability"
Pkg.activate(toktox_path)
Pkg.instantiate()

In [ ]:
venv_python = joinpath(toktox_path, ".venv", "bin", "python")
if !isfile(venv_python)
    run(`$(joinpath(toktox_path, "setup_venv.sh"))`)
end

In [ ]:
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = venv_python

In [ ]:
using Revise
using TokToxStability
using Plots
using DataFrames
using PythonCall

In [ ]:
# only call this once per kernel session
myOFT = TokToxStability.make_oft_env()   


In [ ]:
# An example kinetic EFIT and corresponding geqdsk for DIII-D shot 147131 are provided in cases/D3D
# load_default_settings_tearing sets up GPECSettings for the tearing workflow: ideal-MHD Delta'
# (kinetic_factor=0.0) plus the Glasser/inner-layer inputs (chi_perp, zeff, dc_type, mu_i) that
# compute_glasser_terms needs below.
gpec_settings = TokToxStability.load_default_settings_tearing(1)
kinetic_efit_path = joinpath(pkgdir(TokToxStability), "cases/D3D/g147131.02300_DIIID_KEFIT.kin")
geqdsk_path = joinpath(pkgdir(TokToxStability), "cases/D3D/g147131.02300_DIIID_KEFIT.eqdsk")


In [ ]:
profiles = TokToxStability.read_kinetic_profiles_from_file(kinetic_efit_path)
equilibrium = TokToxStability.read_equilibrium_from_file(geqdsk_path)


In [ ]:
mygs = TokToxStability.setup_tokamaker_from_file!(:D3D, profiles, equilibrium, myOFT)


In [ ]:
# The gEQDSK files created by TokaMaker at each scan iteration can be saved by setting working_directory
# For a file-less option, set working_directory = nothing, which will save the gEQDSKs to a temporary directory that is deleted after the run
working_directory = nothing

# fractions sets the pressure scaling factors to scan for tearing behavior. This example doesn't
# search for the ideal-MHD beta limit itself (see the ideal_beta_limit_from_files/_from_tmtx
# examples for that) -- just pick a fixed set of fractions to see how Delta' evolves with beta_N.
fractions = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

coarse_scan_results = TokToxStability.coarse_scan(mygs, fractions, equilibrium, profiles, gpec_settings; working_directory = working_directory);


In [ ]:
# Creates a DataFrame with the results of the scan 
cols = (:fraction, :beta_n, :dW_total, :dW_plasma, :dW_vacuum, :qmax, :msing)
DataFrame([NamedTuple{cols}(r[c] for c in cols) for r in coarse_scan_results])

In [ ]:
# Each scan point also carries GPEC's classical linear tearing Delta' matrix (ideal-MHD outer-region
# matching, PEST3 convention), since kinetic_factor=0 in gpec_settings above. It's indexed by rational
# surface, not by mode numbers, so singular_surfaces (the (m, n, q) of each row/column) is carried
# alongside it -- see delta_prime_for_mode below.
coarse_scan_results[1].singular_surfaces

## Glasser stabilization (Delta_eff)

The classical Δ' above is the ideal-MHD outer-region jump alone; the Glasser/chi_parallel-matching
correction (Leuthold et al., "ARC physics basis -- MHD", Sec 4.2.1) folds in the resistive-layer
physics to give `Delta_eff = Delta' - Delta_crit`, a more physically complete tearing criterion.
Since `profiles` was read with `read_kinetic_profiles_from_file` (which picked up the `.kin` file's
`wexb(rad/s)` column into `profiles.omega`), every scan point above already carries a
`glasser_terms` entry -- computed once per point, right when `evaluate_factor` still has the raw
GPEC output in hand.

In [ ]:
# Delta_crit and Delta_eff = Delta' - Delta_crit for the 2/1 mode at each scanned point.
[(beta_n=r.beta_n,
  delta_prime_21=TokToxStability.delta_prime_for_mode(r, 2, 1),
  delta_crit_21=TokToxStability.delta_crit_for_mode(r, 2, 1),
  delta_eff_21=TokToxStability.delta_eff_for_mode(r, 2, 1)) for r in coarse_scan_results]

In [ ]:
# Delta' vs Delta_eff (Glasser-stabilized) on one symlog axis, 2/1 mode.
TokToxStability.plot_delta_eff_scan(coarse_scan_results, 2, 1)

In [ ]:
TokToxStability.plot_profiles(coarse_scan_results)
